# 02 · Pré-processamento

Define e valida separações, transformações e remoções das features

In [13]:
# TODO: transformar decisões aprovadas em funções reutilizáveis de src/.

In [2]:
from pathlib import Path
import joblib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
# Definição dos Diretórios
RAW = Path("../data/raw/Steel_industry_data.csv")
PROCESSED = Path("../data/processed")
MODELS = Path("../models")
METADATA = Path("../metadata")

### 1) Importação dos Dados

In [4]:
df = pd.read_csv(RAW)
target_column = "Load_Type"

df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values(by="date").reset_index(drop=True)

df.head()

,date,Usage_kWh,Lagging_Current_Reactive.Power_kVarh,Leading_Current_Reactive_Power_kVarh,CO2(tCO2),Lagging_Current_Power_Factor,Leading_Current_Power_Factor,NSM,WeekStatus,Day_of_week,Load_Type
0,2018-01-01 00:00:00,3.42,3.46,0.0,0.0,70.30,100.0,0,Weekday,Monday,Light_Load
1,2018-01-01 00:15:00,3.17,2.95,0.0,0.0,73.21,100.0,900,Weekday,Monday,Light_Load
2,2018-01-01 00:30:00,4.00,4.46,0.0,0.0,66.77,100.0,1800,Weekday,Monday,Light_Load
3,2018-01-01 00:45:00,3.24,3.28,0.0,0.0,70.28,100.0,2700,Weekday,Monday,Light_Load
4,2018-01-01 01:00:00,3.31,3.56,0.0,0.0,68.09,100.0,3600,Weekday,Monday,Light_Load


### 2) Split Temporal

Os dados serão dividos em treino e teste por meio de um split temporal de 80 % de treino e 20% de teste 

In [5]:
# Definição do Eixo X e do Eixo Y
X = df.drop(columns=[target_column])
y = df[target_column].copy()

# Split temporal
cut = int(len(df) * 0.8)

X_train, X_test = X.iloc[:cut].copy(), X.iloc[cut:].copy()
y_train, y_test = y.iloc[:cut].copy(), y.iloc[cut:].copy()

print(f'Treino: {X_train['date'].min()} -> {X_train['date'].max()}')
print(f'Teste: {X_test['date'].min()} -> {X_test['date'].max()}')

Treino: 2018-01-01 00:00:00 -> 2018-10-19 23:45:00
Teste: 2018-10-20 00:00:00 -> 2018-12-31 23:45:00


### 3) Avaliação das Features

Será adaptado e selecionado as features ideais para avaliar os modelos

In [6]:
def time_feature_adapt(data: pd.DataFrame) -> pd.DataFrame:
    """
    Será realizado a adaptação das colunas relacionadas a tempo, 
    de modo a alinhar melhor a distância entre os pontos
    (não será adpatado a data para dia da semana por 
    apresentar redundância com a coluna 'Day_of_Week')
    """
    out = data.copy()

    out['hour_sin'] = np.sin(2 * np.pi * out["date"].dt.hour / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out["date"].dt.hour / 24)
    out["month_sin"] = np.sin(2 * np.pi * (out["date"].dt.month - 1) / 12)
    out["month_cos"] = np.cos(2 * np.pi * (out["date"].dt.month - 1) / 12)
    #out['NSM_sin'] = np.sin(2 * np.pi * out['NSM'] / 86400)
    #out['NSM_cos'] = np.cos(2 * np.pi * out['NSM'] / 86400)
    
    return out.drop(columns=["date", "NSM"])

X_train_featured = time_feature_adapt(X_train)
X_test_featured = time_feature_adapt(X_test)

#### 3.1) Definição de Features

In [7]:
FEATURE_NUMERIC = [
    'Usage_kWh', # feature central
    'Lagging_Current_Reactive.Power_kVarh',  # r=0.9 com o 'Usage_kWh'
    'Leading_Current_Reactive_Power_kVarh',  # r=-0.32 com o 'Usage_kWh'
    'Lagging_Current_Power_Factor' # r=0.39 com o 'Usage_kWh'
]

FEATURE_HOUR = ['hour_sin', 'hour_cos']
FEATURE_MONTH = ['month_sin', 'month_cos']

FEATURE_CLUSTERS_CATEGORICAL = ['Day_of_week']

FEATURE_DELETED = [
    "date",  # substituída por hour_sin/cos e month_sin/cos
    "NSM",   # substituída por hour_sin/cos (mesma informação, menos granular)
]

### 4) Pré-Processamento

#### 4.1) Início do Pré-Processamento

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), FEATURE_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), FEATURE_CLUSTERS_CATEGORICAL)
    ]
)

pipeline = Pipeline(steps=[
    ("prep", preprocessor),
    ("pca",  PCA(n_components=0.90, random_state=42)),
])

X_train_processed = pipeline.fit_transform(X_train_featured)
X_test_processed = pipeline.transform(X_test_featured) # é utilizado somente 'transform' pata evitar vazamento de dados do teste para o treino

components = pipeline.named_steps["pca"].n_components_

print(f"O Pipeline manteve {components} componetes para explicar 90% da variância")

O Pipeline manteve 6 componetes para explicar 90% da variância


#### 4.2) Checagens

avaliar se as tabelas pré-processadas possuem alguma inconsistência (estrutura diferente, tamanhos diferentes, não apresentam valor)

In [9]:
# Apresenta a mesma forma
assert X_train_processed.shape[1] == X_test_processed.shape[1], \
    "Treino e teste têm números de componentes diferentes"

# Não apresenta valores nulos ou infinitos
assert not np.isnan(X_train_processed).any(), "Há NaN no X_train"
assert not np.isnan(X_test_processed).any(),  "Há NaN no X_test"
assert not np.isinf(X_train_processed).any(), "Há inf no X_train"
assert not np.isinf(X_test_processed).any(),  "Há inf no X_test"

# Split não perdeu linhas
assert len(X_train) + len(X_test) == len(df), "Split perdeu ou duplicou linhas"

#### 4.3) Correlações

Dos Clusters apresentados, será availado o quanto cada um se correlaciona entre si,
teria que ser uma correlação 0 para destacar que cada um agrupa classificações diferentes

In [10]:
comp_df = pd.DataFrame(X_train_processed, columns=[f"PC{i+1}" for i in range(components)])

corr_pos = comp_df.corr().abs()

upper = corr_pos.where(np.triu(np.ones(corr_pos.shape), k=1).astype(bool))
print(f"Correlação máxima entre componentes (deve ser ~0): {upper.stack().max().round(6)}")

Correlação máxima entre componentes (deve ser ~0): 0.0


### 5) Avaliação da Performance dos dados

#### 5.1) Métodos

- o **Coeficiente de Silhueta** para avaliar o quanto cada ponto de dado
está bem agrupado em comparação com os outros grupos vizinhos, caso o resultado
seja abaixo de 0, ou abaixo de 0.2, não há proximidade entre os grupos.

- o **ARI** avalia como o K-Means se comporta diante de várias sementes diferentes (ou em subamostras diferentes),
e se os clusters são os mesmos, se sim são estáveis, se mudam constantemente tem ruído.

- **Estabilidade** 

In [14]:
def silhouette_for(features_train, n_clusters=3, seed=42):
    """ 
    Calculo da Métrica Silhouette através da métrica 'KMeans'
    """
    km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features_train)
    return silhouette_score(features_train, km.labels_)


def silhouette_with(cols, featured, n_clusters=3, seed=42):
    """ 
    Aplicação do padrão StandardScaller para features específicas
    """
    X_scaled = StandardScaler().fit_transform(featured[cols].values)
    return silhouette_for(X_scaled, n_clusters=n_clusters, seed=seed)


def stability_subsample(features, n_clusters=3, n_runs=30, sample_frac=0.8, seed=42):
    """
    Reamostra 80% dos dados, refita o K-Means e mede o ARI
    na interseção de pontos com a solução de referência.
    """
    random_rng = np.random.default_rng(seed=seed)
    km_reference = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features)
    ref_labels = km_reference.labels_
    n = len(features)
    size = int(n * sample_frac)

    list_scores = []
    for _ in range(n_runs):
        index = random_rng.choice(n, size=size, replace=False)
        km = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10).fit(features[index])
        list_scores.append( adjusted_rand_score(ref_labels[index], km.labels_) )

    return float(np.mean(list_scores)), float(np.std(list_scores))



In [ ]:
configs = {
    "Base (Todas as Features)": FEATURE_NUMERIC,
    "Base + hora": FEATURE_NUMERIC + FEATURE_HOUR,
    "Base + mês": FEATURE_NUMERIC + FEATURE_MONTH,
    "Base + hora + mês": FEATURE_NUMERIC + FEATURE_HOUR + FEATURE_MONTH
}

for name, cols in configs.items():
    X_train_featured_scalled = StandardScaler().fit_transform(X_train_featured[cols].values)

    km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train_featured_scalled)
    silhouette_scalled = silhouette_score(X_train_featured_scalled, km.labels_)

    ari_score = adjusted_rand_score(y_train, km.labels_)

    print(f"{name:40s} → silhouette (com StandardScaler) = {silhouette_scalled:.4f} / ARI (Y_train) = {ari_score}")


Base (Todas as Features)                 → silhouette (com StandardScaler) = 0.6361 / ARI (Y_train) = 0.3481584240769701
Base + hora                              → silhouette (com StandardScaler) = 0.4871 / ARI (Y_train) = 0.37392021661234043
Base + mês                               → silhouette (com StandardScaler) = 0.3886 / ARI (Y_train) = 0.3488185093654166
Base + hora + mês                        → silhouette (com StandardScaler) = 0.3500 / ARI (Y_train) = 0.3709590036941739


In [13]:
# Com PCA
silhouette_pca = silhouette_for(features_train=X_train_processed)
print(f"Silhouette com PCA (90% var): {silhouette_pca:.4f}")

Silhouette com PCA (90% var): 0.5409


In [21]:
print('Estabilidade (ARI entre as features)')

for name, cols in configs.items():
    mean_sub, std_sub = stability_subsample(X_train_featured[cols].values)
    print(f'Subamostra 80%: ARI | {name}: Média: {mean_sub} | Desvio Padrão: {std_sub}')
    

Estabilidade (ARI entre as features)
Subamostra 80%: ARI | Base (Todas as Features): Média: 0.997920444395886 | Desvio Padrão: 0.0018346712212086351
Subamostra 80%: ARI | Base + hora: Média: 0.9981615587150254 | Desvio Padrão: 0.0016685573790208347
Subamostra 80%: ARI | Base + mês: Média: 0.9980933397462761 | Desvio Padrão: 0.0018866724580263612
Subamostra 80%: ARI | Base + hora + mês: Média: 0.9981042029141776 | Desvio Padrão: 0.0016243694900679026


#### 5.3) Testes de Remoção de Colunas

- Avaliar se cada cluster enxerga as colunas que foram removidas e se os clusters fazem sentido de domínio.
  
   * 'CO2(tCO2)' -> redundancia com a coluna 'Usage_kWh'
   * 'Leading_Current_Power_Factor' -> suspeita de constância

In [18]:
# Treinamento com as features selecionadas
km_sel = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_train_featured[FEATURE_NUMERIC])
labels_sel = km_sel.labels_

# Coloca as labels novamente com as features antes removidas
df_check = X_train_featured.copy()
df_check['cluster'] = labels_sel

# Calcula a variabilidade das features removidas por cluster
for feature in ['CO2(tCO2)', 'Leading_Current_Power_Factor']:

    mean_per_cluster = df_check.groupby("cluster")[feature].mean()
    variance = mean_per_cluster.max() - mean_per_cluster.min()
    std_global = df_check[feature].std()
    ratio = variance/std_global if std_global > 0 else 0

    print(f"{feature}:")
    print(f"- médias por cluster: {mean_per_cluster.round(3).to_dict()}")
    print(f"- variação entre clusters: {variance:.4f}")
    print(f"- std entre clusters: {std_global:.4f}")
    print(f"- razão variação/std global: {ratio:.3f}\n")


CO2(tCO2):
- médias por cluster: {0: 0.0, 1: 0.033, 2: 0.005}
- variação entre clusters: 0.0326
- std entre clusters: 0.0164
- razão variação/std global: 1.996

Leading_Current_Power_Factor:
- médias por cluster: {0: 99.967, 1: 99.995, 2: 47.842}
- variação entre clusters: 52.1535
- std entre clusters: 30.3782
- razão variação/std global: 1.717



In [29]:
FEATURE_DISCARDED = [
    "CO2(tCO2)", # (alta correlação com 'Usage_kwh' -> redundância quade perfeita); mantida em artifacts para enriquerecer o relatório dos clusters
    #"Leading_Current_Power_Factor", # variância quase nula, inútil para distância, mas útil para o relatório
    "WeekStatus", # redundante com o Day_of_week
]

### 6) Salvamento de Dados

# Processamento de dados (treino e teste)
np.save(PROCESSED / "x_train.npy", X_train_processed)
np.save(PROCESSED / "x_test.npy", X_test_processed)

y_train.to_csv(PROCESSED / "y_train.csv", index=False)
y_test.to_csv(PROCESSED / "y_test.csv", index=False)


# DataFrames Enriquecidos
train_enriched = X_train.copy()
test_enriched = X_test.copy()

for i in range(X_train_processed.shape[1]):
    train_enriched[f"PC{i+1}"] = X_train_processed[:, i]
    test_enriched [f"PC{i+1}"] = X_test_processed[:, i]

train_enriched.to_csv(PROCESSED / "X_train_enriched.csv", index=False)
test_enriched.to_csv(PROCESSED / "X_test_enriched.csv", index=False)


# Pipeline Treinado
joblib.dump(pipeline, MODELS / "preprocessor.joblib")


# Definição das Features
feature_data = {
    "raw_file": str(RAW),
    
    "feature_numeric": FEATURE_NUMERIC,
    "feature_categoric": FEATURE_CLUSTERS_CATEGORICAL,
    "feature_discarded": FEATURE_DISCARDED,
    "feature_deleted": FEATURE_DELETED,
    "target": target_column,

    "pca_components": [f"PC{i+1}" for i in range(components)],
    "one_hot_encoding_features": list(
            pipeline.named_steps["prep"]\
            .named_transformers_["cat"]\
            .get_feature_names_out(FEATURE_CLUSTERS_CATEGORICAL)
    ),

    "split_strategy": "temporal_80_20",
    "n_rows_total": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "train_date_range": [str(X_train["date"].min()), str(X_train["date"].max())],
    "test_date_range":  [str(X_test["date"].min()),  str(X_test["date"].max())],
}

(METADATA / "metadata.json").write_text(
    json.dumps(feature_data, indent=2, ensure_ascii=False),
    encoding="utf-8"
)